## Particle in a Box for Conjugated Dyes

A particle in a box (or particle on a line) is a model used to describe the energy and position of an electron confined in one dimension along a length $L$. The energy levels for the particle are given by

$$
E_n = \frac{n^2 h^2}{8mL^2},\quad n = 1,...,\infty
$$

For a conjugated dye, the particle is assumed to be an electron. 

In this notebook,

* All lengths are entered in units of Å. 
* Energy values are reported in J.


### Setup libraries and constants

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.optimize import brentq
from IPython.display import display, Markdown

In [2]:
# Physical constants (SI)
h = 6.62607015e-34      # Planck constant, J*s
m_e = 9.1093837015e-31  # Electron mass, kg
c = 2.99792458e8        # Speed of light, m/s

ANGSTROM_TO_M = 1.0e-10


### Calculate particle-in-a-box energies and maximum quantum number for a given energy

In [3]:
def pib_energies_joules(length_angstrom, n_levels):
    """Return quantum numbers and 1D PIB energies in Joules."""
    L_m = length_angstrom * ANGSTROM_TO_M
    n = np.arange(1, n_levels + 1)
    E_n = (n**2 * h**2) / (8.0 * m_e * L_m**2)
    return n, E_n


def max_quantum_number_for_energy(length_angstrom, energy_max_j):
    """Largest integer n with E_n <= energy_max_j for a given box length."""
    if energy_max_j <= 0:
        return 0

    L_m = length_angstrom * ANGSTROM_TO_M
    n_max = np.sqrt((8.0 * m_e * (L_m**2) * energy_max_j) / (h**2))
    return int(np.floor(n_max))



### Display particle-in-a-box energies and wavefunctions

In [ ]:
# User controls for the energy-level display
length_widget = widgets.FloatSlider(
    value=12.0,
    min=1.0,
    max=30.0,
    step=0.01,
    description="L (Angstrom)",
    continuous_update=True,
)

energy_max_widget = widgets.FloatLogSlider(
    value=2.0e-18,
    base=10,
    min=-20,
    max=-16,
    step=0.01,
    description="E max (J)",
    readout_format=".2e",
    continuous_update=True,
)

selected_level_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=1,
    step=1,
    description="State n",
    continuous_update=False,
)

show_state_widget = widgets.Checkbox(
    value=False,
    description="Show selected state",
)

view_widget = widgets.ToggleButtons(
    options=[("Wavefunction psi", "psi"), ("Probability |psi|^2", "prob")],
    value="psi",
    description="View",
)


def plot_energy_levels(
    length_angstrom, energy_max_j, selected_n, show_state, view_mode
):
    n_levels = max_quantum_number_for_energy(length_angstrom, energy_max_j)
    if n_levels < 1:
        print("Increase E max: no energy levels fit in the current y-axis range.")
        return

    n, E_n = pib_energies_joules(length_angstrom, n_levels)

    selected_level_widget.max = n_levels

    fig, ax = plt.subplots(figsize=(7, 6))

    # Box is centered at L/2.
    x_left, x_right = 0, length_angstrom
    x_fixed_half_range = (30 - length_angstrom) / 2

    ax.plot([x_left, x_left], [0, energy_max_j], color="black", linewidth=2)
    ax.plot([x_right, x_right], [0, energy_max_j], color="black", linewidth=2)

    for i, energy in enumerate(E_n):
        ax.hlines(energy, x_left, x_right, colors="tab:blue", linewidth=2)
        ax.text(
            min(x_right + 0.35, length_angstrom + 1),
            energy,
            f"n={n[i]}",
            va="center",
            fontsize=9,
        )

    y_top = energy_max_j * 1.1
    if show_state and selected_n is not None:
        x_overlay = np.linspace(x_left, x_right, 1500)
        psi_overlay = infinite_well_wavefunction(
            length_angstrom, selected_n, x_overlay
        )

        if view_mode == "psi":
            max_abs_psi = float(np.max(np.abs(psi_overlay)))
            y_shape = psi_overlay / max_abs_psi if max_abs_psi > 0.0 else psi_overlay
        else:
            prob_overlay = psi_overlay**2
            max_prob = float(np.max(prob_overlay))
            y_shape = prob_overlay / max_prob if max_prob > 0.0 else prob_overlay

        overlay_amplitude = 0.06 * energy_max_j

        y_overlay = E_n[selected_n - 1] + overlay_amplitude * y_shape
        y_top = max(energy_max_j, float(np.max(y_overlay)) * 1.02)
        ax.plot(x_overlay, y_overlay, color="tab:purple", linewidth=1.8, alpha=0.95)

    ax.set_xlim(-x_fixed_half_range, length_angstrom + x_fixed_half_range)
    ax.set_ylim(0.0, y_top)
    ax.set_xlabel("Position (Angstrom)")
    ax.set_ylabel("Energy (J)")
    ax.set_title(
        f"1D Particle-in-a-Box Energy Levels (L = {length_angstrom:.2f} Angstrom, E max = {energy_max_j:.2e} J)"
    )
    plt.show()


def infinite_well_wavefunction(length_angstrom, n_state, x_angstrom):
    """Return normalized finite-well bound-state wavefunction values psi(x)."""

    # x_m = np.asarray(x_angstrom) * ANGSTROM_TO_M

    psi = np.zeros_like(x_angstrom, dtype=float)

    psi = np.sin(n_state * np.pi * x_angstrom / length_angstrom)

    norm = np.trapezoid(psi**2, x_angstrom)
    if norm > 0:
        psi = psi / np.sqrt(norm)

    return psi

levels_out = widgets.Output()


def refresh_levels(*_):
    # sync_finite_selected_level_widget()
    with levels_out:
        levels_out.clear_output(wait=True)
        plot_energy_levels(
            length_widget.value,
            energy_max_widget.value,
            selected_level_widget.value,
            show_state_widget.value,
            view_widget.value,
        )

length_widget.observe(refresh_levels, names="value")
energy_max_widget.observe(refresh_levels, names="value")
selected_level_widget.observe(refresh_levels, names="value")
show_state_widget.observe(refresh_levels, names="value")
view_widget.observe(refresh_levels, names="value")

refresh_levels()

levels_panel = widgets.VBox(
    [
        widgets.HTML("<h3>Finite-Wall Energy Level Display</h3>"),
        widgets.HBox([length_widget, energy_max_widget]),
        widgets.HTML("<p>Use State n to choose a level, then check 'Show selected state' to overlay the selected view on that energy line.</p>"),
        widgets.HBox([selected_level_widget, show_state_widget, view_widget]),
        levels_out,
    ]
)

display(levels_panel)



### Create table of particle-in-a-box energies

In [7]:
def show_energy_table(length_angstrom, energy_max_j):
    n_levels = max_quantum_number_for_energy(length_angstrom, energy_max_j)
    display(Markdown("### Particle in a Box Energy Levels"))
    if n_levels < 1:
        display(Markdown("No levels fall at or below the selected maximum energy."))
        return

    n, E_n = pib_energies_joules(length_angstrom, n_levels)

    lines = ["| n | Energy (J) |", "|---:|---:|"]
    lines.extend([f"| {ni} | {energy:.3e} |" for ni, energy in zip(n, E_n)])

    display(Markdown("\n".join(lines)))


# table_out = widgets.interactive_output(
#     show_energy_table,
#     {"length_angstrom": length_widget, "energy_max_j": energy_max_widget},
# )
# display(table_out)


pib_table_out = widgets.Output()


def refresh_pib_table(*_):
    with pib_table_out:
        pib_table_out.clear_output(wait=True)
        show_energy_table(length_widget.value, energy_max_widget.value)


length_widget.observe(refresh_pib_table, names="value")
energy_max_widget.observe(refresh_pib_table, names="value")

refresh_pib_table()

pib_table_panel = widgets.VBox(
    [
        pib_table_out,
    ]
)

display(pib_table_panel)

### Predict the value of $\lambda_\text{max}$

* Count the number of π-electrons in the dye
* Identify the HOMO and LUMO and the corresponding energies from the table
* Use the code cell to calculate $\Delta E$ (in J) and $\lambda_\text{max}$ (in nm)

In [6]:
# Using 4,4'-dicarbocyanine with 14 π-electrons and a box length of 19.2 Å

n_HOMO = 7
n_LUMO = 8
E_HOMO = 8.008e-19
E_LUMO = 1.046e-18

delta_E = E_LUMO - E_HOMO

lambda_max = (h * c) / delta_E * 1.e9

print("λ_max = ", f"{lambda_max:.2f} nm")

λ_max =  810.13 nm
